In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import concat, row_number, desc
from pyspark.sql.window import Window
from functools import reduce
from typing import List
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable
from pyspark.sql.functions import split, col
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import concat_ws

In [0]:
import os
import sys


In [0]:
cur_dir = os.getcwd()
sys.path.append(cur_dir)


In [0]:
import importlib
import utils.util_silver
importlib.reload(utils.util_silver)
from utils.util_silver import *

In [0]:
import inspect
import utils.util_silver

for name, obj in inspect.getmembers(utils.util_silver, inspect.isclass):
    if obj.__module__ == utils.util_silver.__name__:
        print(f"Class: {name}")
        for func_name, func in inspect.getmembers(obj, inspect.isfunction):
            if func.__module__ == utils.util_silver.__name__:
                print(f"  Function: {func_name}")

In [0]:
df_cust = spark.read.table("uber_pyspark_dbt.bronze.customers")

In [0]:
df_cust = df_cust.withColumn("domain", split(col('email'),'@')[1])

In [0]:
df_cust = df_cust.withColumn("phone_number", regexp_replace(col("phone_number"), r"[^0-9]", ""))

In [0]:
df_cust = df_cust.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
df_cust = df_cust.drop("first_name", "last_name")

In [0]:
cust_obj = silver_transformations(df_cust)

cust_df_trns = cust_obj.silver_dedup(df_cust,['customer_id'],'last_updated_timestamp')

In [0]:
df_cust = cust_obj.silver_process_timestamp(cust_df_trns)
display(df_cust)

In [0]:
if not spark.catalog.tableExists("uber_pyspark_dbt.silver.customers"):
    df_cust.write.format("delta").mode("append").saveAsTable("uber_pyspark_dbt.silver.customers")
else:
    cust_obj.silver_upsert(spark, df_cust,['customer_id'],'customers',['last_updated_timestamp'])

In [0]:
%sql
select count(*) from uber_pyspark_dbt.silver.customers

### Drivers

In [0]:
%sql
select * from uber_pyspark_dbt.bronze.drivers limit 5

In [0]:
df_drivers = spark.read.table("uber_pyspark_dbt.bronze.drivers")
df_drivers = df_drivers.withColumn("phone_number", regexp_replace(col("phone_number"), r"[^0-9]", ""))
df_drivers = df_drivers.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
df_drivers = df_drivers.drop("first_name", "last_name")


In [0]:
# Create class object and apply transformations
drivers_obj = silver_transformations(df_drivers)

df_drivers = drivers_obj.silver_dedup(df_drivers,['driver_id'],'last_updated_timestamp')
df_drivers = drivers_obj.silver_process_timestamp(df_drivers)
display(df_drivers)

In [0]:
# Upsert Logic
if not spark.catalog.tableExists("uber_pyspark_dbt.silver.drivers"):
    df_drivers.write.format("delta").mode("append").saveAsTable("uber_pyspark_dbt.silver.drivers")

else:
    drivers_obj.silver_upsert(spark, df_drivers,['driver_id'],'drivers',['last_updated_timestamp'])

### Locations

In [0]:
%sql
select * from uber_pyspark_dbt.bronze.locations limit 5

In [0]:
# Create class object and apply transformations
df_locations = spark.read.table("uber_pyspark_dbt.bronze.locations")

locations_obj = silver_transformations(df_locations)

df_locations = locations_obj.silver_dedup(df_locations,['location_id'],'last_updated_timestamp')
df_locations = locations_obj.silver_process_timestamp(df_locations)
display(df_locations)

In [0]:
# Upsert Logic
if not spark.catalog.tableExists("uber_pyspark_dbt.silver.locations"):
    df_locations.write.format("delta").mode("append").saveAsTable("uber_pyspark_dbt.silver.locations")

else:
    locations_obj.silver_upsert(spark, df_locations,['location_id'],'locations',['last_updated_timestamp'])

### Payments

In [0]:
%sql
select * from uber_pyspark_dbt.bronze.payments limit 5

In [0]:
# Create class object and apply transformations
df_payments = spark.read.table("uber_pyspark_dbt.bronze.payments")
payments_obj = silver_transformations(df_payments)

df_payments = payments_obj.silver_dedup(df_payments,['payment_id'],'last_updated_timestamp')
df_payments = payments_obj.silver_process_timestamp(df_payments)
display(df_payments)

In [0]:
# Upsert Logic
if not spark.catalog.tableExists("uber_pyspark_dbt.silver.payments"):
    df_payments.write.format("delta").mode("append").saveAsTable("uber_pyspark_dbt.silver.payments")

else:
    payments_obj.silver_upsert(spark, df_payments,['payment_id'],'payments',['last_updated_timestamp'])

### Trips

In [0]:
%sql
select * from uber_pyspark_dbt.bronze.trips limit 5

In [0]:
# Create class object and apply transformations
df_trips = spark.read.table("uber_pyspark_dbt.bronze.trips")
trips_obj = silver_transformations(df_trips)

df_trips = trips_obj.silver_dedup(df_trips,['trip_id'],'last_updated_timestamp')
df_trips = trips_obj.silver_process_timestamp(df_trips)
display(df_trips)

In [0]:
# Upsert Logic
if not spark.catalog.tableExists("uber_pyspark_dbt.silver.trips"):
    df_trips.write.format("delta").mode("append").saveAsTable("uber_pyspark_dbt.silver.trips")

else:
    trips_obj.silver_upsert(spark, df_trips,['trip_id'],'trips',['last_updated_timestamp'])

### Vehicles

In [0]:
%sql
select * from uber_pyspark_dbt.bronze.vehicles limit 5

In [0]:
# Create class object and apply transformations
df_vehicles = spark.read.table("uber_pyspark_dbt.bronze.vehicles")
vehicles_obj = silver_transformations(df_vehicles)

df_vehicles = vehicles_obj.silver_dedup(df_vehicles,['vehicle_id'],'last_updated_timestamp')
df_vehicles = vehicles_obj.silver_process_timestamp(df_vehicles)
display(df_vehicles)

In [0]:
# Upsert Logic
if not spark.catalog.tableExists("uber_pyspark_dbt.silver.vehicles"):
    df_vehicles.write.format("delta").mode("append").saveAsTable("uber_pyspark_dbt.silver.vehicles")

else:
    vehicles_obj.silver_upsert(spark, df_vehicles,['vehicle_id'],'vehicles',['last_updated_timestamp'])